In [27]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np

# ---------------------------------------------------------
# 1. Hardware Configuration
# ---------------------------------------------------------
# This automatically detects the Kaggle GPU if enabled
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Training on device: {device}")

# ---------------------------------------------------------
# 2. Data Loading
# ---------------------------------------------------------
transform = transforms.ToTensor()

train_dataset = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
train_loader = DataLoader(dataset=train_dataset, batch_size=128, shuffle=True)

test_dataset = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)
test_loader = DataLoader(dataset=test_dataset, batch_size=1000, shuffle=False)

# ---------------------------------------------------------
# 3. Model Definition
# ---------------------------------------------------------
class SphericalAutoencoder(nn.Module):
    def __init__(self):
        super(SphericalAutoencoder, self).__init__()
        self.encoder = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, 128),
            nn.ReLU(),
            nn.Linear(128, 36),
            nn.ReLU(),
            nn.Linear(36, 3) # 3D latent space
        )
        
        self.decoder = nn.Sequential(
            nn.Linear(3, 36),
            nn.ReLU(),
            nn.Linear(36, 128),
            nn.ReLU(),
            nn.Linear(128, 28 * 28),
            nn.Sigmoid() 
        )

    def encode(self, x):
        h = self.encoder(x)
        z = F.normalize(h, p=2, dim=1) # L2 Projection to unit sphere
        return z

    def forward(self, x):
        z = self.encode(x)
        reconstructed = self.decoder(z)
        return reconstructed.view(-1, 1, 28, 28)

# ---------------------------------------------------------
# 4. Initialization & Training Loop
# ---------------------------------------------------------
# Move the model architecture to the GPU
model = SphericalAutoencoder().to(device)

criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

num_epochs = 10
print("Starting training...")

for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    
    # We only need the images for an autoencoder, so we ignore labels in training using '_'
    for images, _ in train_loader:
        # Move images to the GPU
        images = images.to(device)
        
        optimizer.zero_grad()
        reconstructed = model(images)
        loss = criterion(reconstructed, images)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
    avg_loss = total_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}")

# ---------------------------------------------------------
# 5. Inference & Visualization
# ---------------------------------------------------------
print("Generating 3D visualization...")
model.eval()

latent_vectors = []
latent_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        # Move test batches to GPU for encoding
        images = images.to(device) 
        
        z = model.encode(images)
        
        # Move the resulting vectors back to the CPU so numpy/matplotlib can read them
        latent_vectors.append(z.cpu().numpy())
        latent_labels.append(labels.numpy())

# Flatten lists into single numpy arrays
latent_vectors = np.concatenate(latent_vectors, axis=0)
latent_labels = np.concatenate(latent_labels, axis=0)



Training on device: cuda
Starting training...
Epoch [1/10], Loss: 0.2570
Epoch [2/10], Loss: 0.2003
Epoch [3/10], Loss: 0.1955
Epoch [4/10], Loss: 0.1922
Epoch [5/10], Loss: 0.1900
Epoch [6/10], Loss: 0.1874
Epoch [7/10], Loss: 0.1854
Epoch [8/10], Loss: 0.1840
Epoch [9/10], Loss: 0.1822
Epoch [10/10], Loss: 0.1813
Generating 3D visualization...


In [38]:
# ---------------------------------------------------------
# 6. Interactive Plotting with Plotly (Filtered)
# ---------------------------------------------------------
import plotly.express as px
import pandas as pd

print("Generating filtered 3D visualization...")

# Pack our data into a DataFrame
df = pd.DataFrame({
    'x': latent_vectors[:, 0],
    'y': latent_vectors[:, 1],
    'z': latent_vectors[:, 2],
    'Digit': latent_labels.astype(str) 
})

# --- NEW: Filter for specific digits ---
# Choose exactly which 3 digits you want to isolate
selected_digits = ['0', '1', '2', '3', '8']
filtered_df = df[df['Digit'].isin(selected_digits)]

# Create the interactive 3D scatter plot using ONLY the filtered data
fig = px.scatter_3d(
    filtered_df, 
    x='x', 
    y='y', 
    z='z',
    color='Digit',
    color_discrete_sequence=px.colors.qualitative.Plotly, 
    opacity=0.7,
    title=f"Filtered 3D Spherical Latent Space (Showing: {', '.join(selected_digits)})"
)

# Shrink the marker size so the clusters are well-defined
fig.update_traces(marker=dict(size=2))

# Force the axis aspect ratio to be exactly 1:1:1
fig.update_layout(scene=dict(aspectmode='cube'))
# Display the plot in your Kaggle notebook
fig.show()

Generating filtered 3D visualization...


In [31]:
# Save the interactive figure as an HTML file
fig.write_html("spherical_latent_space.html")

print("Interactive figure saved as HTML!")

Interactive figure saved as HTML!


In [40]:
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
import numpy as np

print("Generating rotating GIF... (This might take a minute)")

# 1. Set up the figure and 3D axis
fig = plt.figure(figsize=(8, 8), facecolor='white')
ax = fig.add_subplot(111, projection='3d')

# 2. Plot the static data
# (Assuming latent_vectors and latent_labels are still in memory from earlier)
scatter = ax.scatter(
    latent_vectors[:, 0], 
    latent_vectors[:, 1], 
    latent_vectors[:, 2], 
    c=latent_labels, 
    cmap='tab10', 
    s=5, 
    alpha=0.8
)

# Clean up the visual presentation
ax.set_xticks([])
ax.set_yticks([])
ax.set_zticks([])
ax.set_title("3D Spherical Latent Space", pad=20)

# Remove the gray background panes for a cleaner look
ax.xaxis.set_pane_color((1.0, 1.0, 1.0, 0.0))
ax.yaxis.set_pane_color((1.0, 1.0, 1.0, 0.0))
ax.zaxis.set_pane_color((1.0, 1.0, 1.0, 0.0))
ax.grid(False)

# 3. Define the animation function
# This function is called for every frame. We change the 'azim' (azimuthal viewing angle)
def update(frame_angle):
    # elev = camera height angle, azim = camera rotation angle
    ax.view_init(elev=20, azim=frame_angle)
    return scatter,

# 4. Create the animation
# We rotate from 0 to 360 degrees in steps of 4 degrees (90 total frames)
ani = FuncAnimation(
    fig, 
    update, 
    frames=np.arange(0, 360, 4), 
    interval=50 # 50 milliseconds per frame (20 frames per second)
)

# 5. Save the animation as a GIF
# PillowWriter is built into matplotlib and doesn't require extra Kaggle installations
gif_filename = "spherical_latent_space.gif"
ani.save(gif_filename, writer=PillowWriter(fps=20))

print(f"Success! Saved as {gif_filename}")

# Close the plot so it doesn't display a static duplicate in the notebook
plt.close(fig)

Generating rotating GIF... (This might take a minute)
Success! Saved as spherical_latent_space.gif
